In [1]:

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

#### State Backend

In [2]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:llama-3.3-70b-versatile")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000014D02196A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000014D022CFE60>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
# 1. Create the agent - these two are equivalent

agent = create_deep_agent(model=model)

## Under the hood, this is what 'agent' is doing - explicit StateBackend

agent2 = create_deep_agent(model=model, backend=StateBackend())

In [5]:
# Invoke the agent and ask it to WRITE a file
# (StateBackend keeps the file inside LangGraph State)

result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record Video\n2. Edit Video\n3. Upload Video\n"
            "Then tell me you've done it"
        )
    }]
})

In [8]:
print("\n--- Agent reply ----------------")
print(result["messages"][-1].content)


--- Agent reply ----------------
I've created the file at /notes/todo.txt with the specified content.


In [9]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record Video\n2. Edit Video\n3. Upload Video\nThen tell me you've done it", additional_kwargs={}, response_metadata={}, id='c7cea7e2-dd1e-4b0c-9862-0a3f55ff6916'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '0z44htdkw', 'function': {'arguments': '{"content":"1. Record Video\\n2. Edit Video\\n3. Upload Video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 9160, 'total_tokens': 9197, 'completion_time': 0.130777814, 'completion_tokens_details': None, 'prompt_time': 0.462083785, 'prompt_tokens_details': None, 'queue_time': 0.052685522, 'total_time': 0.592861599}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9c

In [10]:
# -------------------------------------------------------------------
# 3. Check the backend is working
# With StateBackend, written files appear under result["files"]
# -------------------------------------------------------------------

print("\n ------------- Backend Check ---------------")
files = result.get("files", {})

if files:
    print(f" StateBackend is working - {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n {path}\n { '-' * 40} \n {content}")
else:
    print(" No files found in state. Either the agent didn't write a file, or the backend isn't wired up correctly")


 ------------- Backend Check ---------------
 StateBackend is working - 1 file(s) in state:

 /notes/todo.txt
 ---------------------------------------- 
 {'content': '1. Record Video\n2. Edit Video\n3. Upload Video', 'encoding': 'utf-8', 'created_at': '2026-07-26T03:08:07.069779+00:00', 'modified_at': '2026-07-26T03:08:07.069779+00:00'}


#### FileSystem Backend (local disk)

In [15]:
# 1. Create the agent with a real-disk backend
# root-dir = '.' -> files land relative to your current working directory
# virtual_mode = True -> agent uses virtual paths like /notes/toto.txt, mappen onto root_dir

from deepagents.backends import FilesystemBackend

ROOT = '.'
agent3 = create_deep_agent(model=model, backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True))
print(f" Agent created with FileSystemBackend(root_dir = {ROOT})")
print("Files written by agent wull appear on your actual disk")

 Agent created with FileSystemBackend(root_dir = .)
Files written by agent wull appear on your actual disk


In [17]:
# Invoke the agent and ask it to WRITE a file
# (StateBackend keeps the file inside LangGraph State)

result = agent3.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record Video\n2. Edit Video\n3. Upload Video\n"
            "Then tell me you've done it"
        )
    }]
})

print("\n--- Agent reply ----------------")
print(result["messages"][-1].content)


--- Agent reply ----------------
I've created the file at /notes/todo.txt with the specified content.


In [18]:
# ---------------------------------------------------------------------
# 3. CHECK the backend is working - look on the REAL disk
# with virtual mode = True, /notes/toto.txt maps to ./notes/todo.txt
# ---------------------------------------------------------------------

from pathlib import Path
print("\n --------------- Backend Check (real filesystem) ----------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(" FileSystem Backend is working - file exists on disk")
    print(f"{disk_path.resolve()}\n{'-'*40}")
    print(disk_path.read_text())
else:
    print(f" Expected file not found at {disk_path.resolve()}")


 --------------- Backend Check (real filesystem) ----------------
 FileSystem Backend is working - file exists on disk
C:\Users\harsh\Desktop\Udemy Courses\deepagentscourse\deepagentsdemo\notes\todo.txt
----------------------------------------
1. Record Video
2. Edit Video
3. Upload Video


#### Deep Agent - StoreBackend verification

Creates a deep agent backed by a LangGraph store, invokes it to write a file on one thread, then proves the backend works by reading that files back on a DIFFERENT thread - something StateBackend cannot do.

In [24]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend

store = InMemoryStore()

agent4 = create_deep_agent(
    model="groq:qwen/qwen3.6-27b",
    backend=StoreBackend(
        # Local dev: static namespace, No deployment runtime neeeded.
        # In a Langsmith Deployment yu'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store
)

print(" Agent created with StoreBackend + static namespace")

 Agent created with StoreBackend + static namespace


In [25]:
import os
import uuid

# ---------------------------------------------------------------
# 2. WRITE a file
# ---------------------------------------------------------------

thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent4.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo1.txt with exactly this content:\n"
                "1. Record Video\n2. Edit Video\n3. Upload Video\n"
            )
        }]
    },
    config=thread_1
)

print("\n -------- Agent Reply ---------------")
print(result["messages"][-1].content)


 -------- Agent Reply ---------------
Created `/notes/todo1.txt` with the specified content.


In [26]:
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent4.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Read /notes/todo1.txt back to me verbatim"
        }]
    },
    config=thread_2
)

print("\n -------- Agent Reply from a different thread ---------------")
print(result["messages"][-1].content)


 -------- Agent Reply from a different thread ---------------
1. Record Video
2. Edit Video
3. Upload Video
